# Image Captioning with PyTorch: ResNet Encoder + Attention Decoder

This notebook trains and evaluates an image captioning model on Flickr8k.
Model definitions and utilities live in `models.py` and `utils.py`;
paths and W&B settings are in `global_config.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

import json
import pickle
import random
import time
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import ResNet50_Weights

import global_config as gc
from models import (
    DecoderRNNWithAttention,
    EncoderCNN,
    Flickr8kCaptionDataset,
    Vocabulary,
    make_collate_fn,
    train_one_epoch,
    validate_one_epoch,
)
from utils import (
    clean_caption,
    clean_reference_caption,
    device_check,
    evaluate_bleu,
    generate_caption_beam_search,
    generate_caption_greedy,
    load_captions_file,
    save_checkpoint,
    show_image_with_captions,
    show_prediction_pytorch,
)

In [ ]:
# Reproducibility settings.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = device_check()

# Verify data paths.
assert gc.DATA_DIR.exists(), (
    f"Dataset directory not found: {gc.DATA_DIR}"
)
assert gc.IMAGE_DIR.exists(), (
    f"Image directory not found: {gc.IMAGE_DIR}"
)
assert gc.CAPTIONS_FILE.exists(), (
    f"Captions file not found: {gc.CAPTIONS_FILE}"
)

# Create a timestamped run directory.
RUN_NAME = (
    "pytorch_resnet_attention_wandb_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)
RUN_DIR = gc.RUNS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("Run directory:", RUN_DIR.resolve())

## Loading and cleaning captions

Each image in Flickr8k has multiple reference captions.
We lowercase, strip punctuation, and add `<start>` / `<end>` tokens.

In [ ]:
captions_df = load_captions_file(gc.CAPTIONS_FILE)
captions_df["clean_caption"] = captions_df["caption"].apply(
    clean_caption
)

print("Number of caption rows:", len(captions_df))
print("Number of unique images:", captions_df["image"].nunique())

# Build image-to-captions dict; ignore missing image files.
image_to_captions = defaultdict(list)

for _, row in captions_df.iterrows():
    image_name = row["image"]
    image_path = gc.IMAGE_DIR / image_name

    if image_path.exists():
        image_to_captions[image_name].append(row["clean_caption"])

image_to_captions = dict(image_to_captions)
all_image_names = sorted(list(image_to_captions.keys()))

print("Images with valid captions:", len(all_image_names))

sample_image = all_image_names[0]
print("Sample image:", sample_image)
print("Sample captions:")
for caption in image_to_captions[sample_image]:
    print("-", caption)

In [ ]:
show_image_with_captions(sample_image, image_to_captions, gc.IMAGE_DIR)

## Train, validation, and test split

The dataset is split by image filename, not by caption row,
to prevent data leakage between splits.

In [ ]:
# Split by image names.
train_images, temp_images = train_test_split(
    all_image_names,
    test_size=0.20,
    random_state=SEED,
)

val_images, test_images = train_test_split(
    temp_images,
    test_size=0.50,
    random_state=SEED,
)

print("Train images:", len(train_images))
print("Validation images:", len(val_images))
print("Test images:", len(test_images))

split_info = {
    "train_images": train_images,
    "val_images": val_images,
    "test_images": test_images,
}

with open(RUN_DIR / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Saved split information to:", RUN_DIR / "split_info.json")

## Vocabulary

The vocabulary is built only from training captions.
Special tokens: `<pad>`, `<unk>`, `<start>`, `<end>`.
Words appearing fewer than `min_freq=5` times are excluded.

In [ ]:
train_captions = []
for image_name in train_images:
    train_captions.extend(image_to_captions[image_name])

vocab = Vocabulary(min_freq=5)
vocab.build(train_captions)

pad_idx = vocab.word2idx[vocab.pad_token]
unk_idx = vocab.word2idx[vocab.unk_token]
start_idx = vocab.word2idx[vocab.start_token]
end_idx = vocab.word2idx[vocab.end_token]

print("Vocabulary size:", len(vocab))
print("PAD index:", pad_idx)
print("UNK index:", unk_idx)
print("START index:", start_idx)
print("END index:", end_idx)

with open(RUN_DIR / "vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

print("Saved vocabulary to:", RUN_DIR / "vocab.pkl")

In [ ]:
# Inspect a sample numericalized caption.
sample_caption = image_to_captions[sample_image][0]
sample_ids = vocab.numericalize(sample_caption)

print("Sample caption:")
print(sample_caption)

print("\nToken IDs:")
print(sample_ids)

print("\nDecoded caption:")
print(vocab.decode_ids(sample_ids, remove_special=False))

## Image transformations

The encoder uses a pretrained ResNet50 model.
Training images receive augmentation; validation/test images do not.

In [ ]:
# Image preprocessing for pretrained ResNet50.
resnet_weights = ResNet50_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=resnet_weights.transforms().mean,
        std=resnet_weights.transforms().std,
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=resnet_weights.transforms().mean,
        std=resnet_weights.transforms().std,
    ),
])

## Dataset and DataLoader

Each dataset item contains:
- an image tensor
- one caption (padded to batch length)
- the image filename

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 0

collate_fn = make_collate_fn(pad_idx)

train_dataset = Flickr8kCaptionDataset(
    image_names=train_images,
    captions_mapping=image_to_captions,
    image_dir=gc.IMAGE_DIR,
    vocab=vocab,
    transform=train_transform,
    random_caption=True,
)

val_dataset = Flickr8kCaptionDataset(
    image_names=val_images,
    captions_mapping=image_to_captions,
    image_dir=gc.IMAGE_DIR,
    vocab=vocab,
    transform=eval_transform,
    random_caption=False,
)

test_dataset = Flickr8kCaptionDataset(
    image_names=test_images,
    captions_mapping=image_to_captions,
    image_dir=gc.IMAGE_DIR,
    vocab=vocab,
    transform=eval_transform,
    random_caption=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# Test one batch.
images, captions, lengths, image_names_batch = next(
    iter(train_loader)
)

print("Images shape:", images.shape)
print("Captions shape:", captions.shape)
print("Lengths shape:", lengths.shape)
print("First image name:", image_names_batch[0])
print("First caption IDs:", captions[0].tolist())
print("First caption decoded:")
print(vocab.decode_ids(captions[0].tolist(), remove_special=False))

## Model architecture

The model has three main components:
1. **EncoderCNN** — pretrained ResNet50 producing spatial feature maps
2. **Attention** — additive (Bahdanau) attention over encoder features
3. **DecoderRNNWithAttention** — LSTM decoder that attends to image regions

See `models.py` for the full implementations.

In [ ]:
# Model hyperparameters.
ENCODED_IMAGE_SIZE = 14
ENCODER_DIM = 2048
ATTENTION_DIM = 512
EMBED_DIM = 256
DECODER_DIM = 512
DROPOUT = 0.5

encoder = EncoderCNN(
    encoded_image_size=ENCODED_IMAGE_SIZE,
    fine_tune=False,
).to(device)

decoder = DecoderRNNWithAttention(
    attention_dim=ATTENTION_DIM,
    embed_dim=EMBED_DIM,
    decoder_dim=DECODER_DIM,
    vocab_size=len(vocab),
    encoder_dim=ENCODER_DIM,
    dropout=DROPOUT,
).to(device)

model_config = {
    "encoded_image_size": ENCODED_IMAGE_SIZE,
    "encoder_dim": ENCODER_DIM,
    "attention_dim": ATTENTION_DIM,
    "embed_dim": EMBED_DIM,
    "decoder_dim": DECODER_DIM,
    "dropout": DROPOUT,
    "vocab_size": len(vocab),
}

print(encoder)
print(decoder)

In [ ]:
# Test a forward pass with one batch.
encoder.eval()
decoder.eval()

images = images.to(device)
captions = captions.to(device)
lengths = lengths.to(device)

with torch.no_grad():
    encoder_out = encoder(images)
    (
        predictions,
        sorted_captions,
        decode_lengths,
        alphas,
        sort_ind,
    ) = decoder(encoder_out, captions, lengths)

print("Encoder output shape:", encoder_out.shape)
print("Predictions shape:", predictions.shape)
print("Attention weights shape:", alphas.shape)
print("Number of decode lengths:", len(decode_lengths))
print("First decode length:", decode_lengths[0])

## Loss function and optimization

The model is trained with cross-entropy loss.
The encoder uses a separate, lower learning rate when fine-tuned.

In [ ]:
# Training hyperparameters.
LEARNING_RATE = 4e-4
ENCODER_LEARNING_RATE = 1e-4
GRAD_CLIP = 5.0
ALPHA_C = 1.0

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

decoder_optimizer = optim.Adam(
    params=filter(
        lambda p: p.requires_grad, decoder.parameters()
    ),
    lr=LEARNING_RATE,
)

encoder_optimizer = None

if any(param.requires_grad for param in encoder.parameters()):
    encoder_optimizer = optim.Adam(
        params=filter(
            lambda p: p.requires_grad, encoder.parameters()
        ),
        lr=ENCODER_LEARNING_RATE,
    )

print("Decoder optimizer:", decoder_optimizer)
print("Encoder optimizer:", encoder_optimizer)

## Weights & Biases experiment tracking

This experiment uses Weights & Biases to log training metrics
and generated captions.

In [ ]:
wandb_config = {
    "architecture": "ResNet50 Encoder + Attention + LSTM Decoder",
    "dataset": "Flickr8k",
    "framework": "PyTorch",
    "fusion_method": "Attention",
    "encoded_image_size": ENCODED_IMAGE_SIZE,
    "encoder_dim": ENCODER_DIM,
    "attention_dim": ATTENTION_DIM,
    "embed_dim": EMBED_DIM,
    "decoder_dim": DECODER_DIM,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "learning_rate": LEARNING_RATE,
    "encoder_learning_rate": ENCODER_LEARNING_RATE,
    "grad_clip": GRAD_CLIP,
    "alpha_c": ALPHA_C,
    "vocab_size": len(vocab),
    "min_word_freq": vocab.min_freq,
    "train_images": len(train_images),
    "val_images": len(val_images),
    "test_images": len(test_images),
    "seed": SEED,
    "num_epochs": 50,
}

wandb_run = wandb.init(
    project=gc.WANDB_PROJECT,
    entity=gc.WANDB_ENTITY,
    name=RUN_NAME,
    config=wandb_config,
    dir=str(gc.PROJECT_DIR),
    job_type="training",
)

wandb.watch(decoder, log="gradients", log_freq=100)

print("W&B run initialized:")
print("Run name:", wandb_run.name)
print("Run URL:", wandb_run.url)

## Training loop

The encoder is frozen at first.
This makes training faster and more stable in the early stages.

In [ ]:
NUM_EPOCHS = 50
BEST_VAL_LOSS = float("inf")

history = {
    "epoch": [],
    "train_loss": [],
    "train_top5": [],
    "val_loss": [],
    "val_top5": [],
}

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    start_time = time.time()

    train_loss, train_top5 = train_one_epoch(
        encoder=encoder,
        decoder=decoder,
        train_loader=train_loader,
        criterion=criterion,
        decoder_optimizer=decoder_optimizer,
        encoder_optimizer=encoder_optimizer,
        device=device,
        grad_clip=GRAD_CLIP,
        alpha_c=ALPHA_C,
    )

    val_loss, val_top5 = validate_one_epoch(
        encoder=encoder,
        decoder=decoder,
        val_loader=val_loader,
        criterion=criterion,
        device=device,
        alpha_c=ALPHA_C,
    )

    epoch_time = time.time() - start_time

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_top5"].append(train_top5)
    history["val_loss"].append(val_loss)
    history["val_top5"].append(val_top5)

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        RUN_DIR / "training_history.csv", index=False
    )

    is_best = val_loss < BEST_VAL_LOSS

    if is_best:
        BEST_VAL_LOSS = val_loss

    wandb.log(
        {
            "epoch": epoch,
            "train/loss": train_loss,
            "train/top5_accuracy": train_top5,
            "val/loss": val_loss,
            "val/top5_accuracy": val_top5,
            "time/epoch_minutes": epoch_time / 60,
            "best/val_loss": BEST_VAL_LOSS,
        },
        step=epoch,
    )

    save_checkpoint(
        run_dir=RUN_DIR,
        epoch=epoch,
        encoder=encoder,
        decoder=decoder,
        decoder_optimizer=decoder_optimizer,
        encoder_optimizer=encoder_optimizer,
        best_val_loss=BEST_VAL_LOSS,
        history=history,
        vocab=vocab,
        model_config=model_config,
        is_best=is_best,
    )

    print(
        f"Epoch {epoch} completed in {epoch_time / 60:.2f} minutes | "
        f"train_loss={train_loss:.4f}, train_top5={train_top5:.2f}, "
        f"val_loss={val_loss:.4f}, val_top5={val_top5:.2f}"
    )

    if is_best:
        print("New best checkpoint saved.")

print("Training finished.")
print("Latest checkpoint:", RUN_DIR / "checkpoint_latest.pth")
print("Best checkpoint:", RUN_DIR / "checkpoint_best.pth")

## Training curves

In [ ]:
history_df = pd.read_csv(RUN_DIR / "training_history.csv")

history_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"], history_df["train_loss"], label="train_loss"
)
plt.plot(
    history_df["epoch"], history_df["val_loss"], label="val_loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"], history_df["train_top5"],
    label="train_top5"
)
plt.plot(
    history_df["epoch"], history_df["val_top5"], label="val_top5"
)
plt.xlabel("Epoch")
plt.ylabel("Top-5 Accuracy (%)")
plt.title("Training and Validation Top-5 Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Caption generation

The model generates captions using greedy decoding:
at each step, the highest-probability token is selected.

In [ ]:
# Show predictions for a few test images.
for image_name in test_images[:10]:
    show_prediction_pytorch(
        image_name=image_name,
        encoder=encoder,
        decoder=decoder,
        vocab=vocab,
        device=device,
        image_to_captions=image_to_captions,
        image_dir=gc.IMAGE_DIR,
        dataset_transform=eval_transform,
    )

In [ ]:
# Generate text-only predictions for easier copying and sharing.
NUM_EXAMPLES = 10
output_lines = []

for i, image_name in enumerate(
    test_images[:NUM_EXAMPLES], start=1
):
    image_path = gc.IMAGE_DIR / image_name
    raw_image = Image.open(image_path).convert("RGB")
    model_image = eval_transform(raw_image)

    generated_caption = generate_caption_greedy(
        encoder=encoder,
        decoder=decoder,
        image=model_image,
        vocab=vocab,
        device=device,
        max_length=30,
    )

    output_lines.append("=" * 80)
    output_lines.append(f"Example {i}")
    output_lines.append(f"Image: {image_name}")
    output_lines.append("")
    output_lines.append("Generated caption:")
    output_lines.append(generated_caption)
    output_lines.append("")
    output_lines.append("Reference captions:")

    for caption in image_to_captions[image_name]:
        clean_ref = clean_reference_caption(caption)
        output_lines.append(f"- {clean_ref}")

    output_lines.append("")

text_output = "\n".join(output_lines)
print(text_output)

predictions_path = RUN_DIR / "sample_predictions_text.txt"

with open(predictions_path, "w", encoding="utf-8") as f:
    f.write(text_output)

print("\nSaved text predictions to:", predictions_path)

## Load the best checkpoint

The best checkpoint is selected based on validation loss.

In [ ]:
BEST_CHECKPOINT_PATH = RUN_DIR / "checkpoint_best.pth"

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

encoder.load_state_dict(checkpoint["encoder_state_dict"])
decoder.load_state_dict(checkpoint["decoder_state_dict"])

encoder.to(device)
decoder.to(device)

print("Loaded best checkpoint from:", BEST_CHECKPOINT_PATH)
print("Best validation loss:", checkpoint["best_val_loss"])
print("Checkpoint epoch:", checkpoint["epoch"])

## Beam search decoding

Greedy decoding selects the most likely word at each step.
Beam search maintains the top-k hypotheses, which often yields
higher quality captions at the cost of additional computation.

In [ ]:
# Compare greedy decoding and beam search on test examples.
NUM_EXAMPLES = 10
BEAM_SIZE = 3

output_lines = []

for i, image_name in enumerate(
    test_images[:NUM_EXAMPLES], start=1
):
    image_path = gc.IMAGE_DIR / image_name
    raw_image = Image.open(image_path).convert("RGB")
    model_image = eval_transform(raw_image)

    greedy_caption = generate_caption_greedy(
        encoder=encoder,
        decoder=decoder,
        image=model_image,
        vocab=vocab,
        device=device,
        max_length=30,
    )

    beam_caption = generate_caption_beam_search(
        encoder=encoder,
        decoder=decoder,
        image=model_image,
        vocab=vocab,
        device=device,
        beam_size=BEAM_SIZE,
        max_length=30,
    )

    output_lines.append("=" * 80)
    output_lines.append(f"Example {i}")
    output_lines.append(f"Image: {image_name}")
    output_lines.append("")
    output_lines.append("Greedy caption:")
    output_lines.append(greedy_caption)
    output_lines.append("")
    output_lines.append(f"Beam search caption, beam_size={BEAM_SIZE}:")
    output_lines.append(beam_caption)
    output_lines.append("")
    output_lines.append("Reference captions:")

    for caption in image_to_captions[image_name]:
        clean_ref = clean_reference_caption(caption)
        output_lines.append(f"- {clean_ref}")

    output_lines.append("")

text_output = "\n".join(output_lines)
print(text_output)

beam_path = (
    RUN_DIR / "sample_predictions_greedy_vs_beam.txt"
)

with open(beam_path, "w", encoding="utf-8") as f:
    f.write(text_output)

print("\nSaved greedy vs beam predictions to:", beam_path)

## Quantitative evaluation with BLEU

BLEU evaluates n-gram overlap between generated and reference captions.

In [ ]:
# Log 5 generated test examples to Weights & Biases.
NUM_WANDB_EXAMPLES = 5
BEAM_SIZE = 3

caption_examples_table = wandb.Table(
    columns=[
        "image",
        "image_name",
        "greedy_caption",
        "beam_caption",
        "reference_captions",
    ]
)

for image_name in test_images[:NUM_WANDB_EXAMPLES]:
    image_path = gc.IMAGE_DIR / image_name
    raw_image = Image.open(image_path).convert("RGB")
    model_image = eval_transform(raw_image)

    greedy_caption = generate_caption_greedy(
        encoder=encoder,
        decoder=decoder,
        image=model_image,
        vocab=vocab,
        device=device,
        max_length=30,
    )

    beam_caption = generate_caption_beam_search(
        encoder=encoder,
        decoder=decoder,
        image=model_image,
        vocab=vocab,
        device=device,
        beam_size=BEAM_SIZE,
        max_length=30,
    )

    references = [
        clean_reference_caption(caption)
        for caption in image_to_captions[image_name]
    ]

    caption_examples_table.add_data(
        wandb.Image(str(image_path)),
        image_name,
        greedy_caption,
        beam_caption,
        " | ".join(references),
    )

wandb.log({"examples/generated_captions": caption_examples_table})
print("Logged generated caption examples to W&B.")

In [ ]:
# Evaluate both greedy and beam search decoding.
greedy_bleu = evaluate_bleu(
    encoder=encoder,
    decoder=decoder,
    image_names=test_images,
    image_to_captions=image_to_captions,
    vocab=vocab,
    device=device,
    image_dir=gc.IMAGE_DIR,
    eval_transform=eval_transform,
    decoding_method="greedy",
    max_length=30,
)

beam_bleu = evaluate_bleu(
    encoder=encoder,
    decoder=decoder,
    image_names=test_images,
    image_to_captions=image_to_captions,
    vocab=vocab,
    device=device,
    image_dir=gc.IMAGE_DIR,
    eval_transform=eval_transform,
    decoding_method="beam",
    beam_size=3,
    max_length=30,
)

bleu_results = pd.DataFrame([
    {"decoding": "greedy", **greedy_bleu},
    {"decoding": "beam_size_3", **beam_bleu},
])

bleu_results.to_csv(RUN_DIR / "bleu_results.csv", index=False)

bleu_results

In [ ]:
wandb.log({
    "test/greedy_BLEU_1": greedy_bleu["BLEU-1"],
    "test/greedy_BLEU_2": greedy_bleu["BLEU-2"],
    "test/greedy_BLEU_3": greedy_bleu["BLEU-3"],
    "test/greedy_BLEU_4": greedy_bleu["BLEU-4"],
    "test/beam_BLEU_1": beam_bleu["BLEU-1"],
    "test/beam_BLEU_2": beam_bleu["BLEU-2"],
    "test/beam_BLEU_3": beam_bleu["BLEU-3"],
    "test/beam_BLEU_4": beam_bleu["BLEU-4"],
})

wandb.summary["best_val_loss"] = checkpoint["best_val_loss"]
wandb.summary["best_checkpoint_epoch"] = checkpoint["epoch"]
wandb.summary["best_checkpoint_path"] = str(BEST_CHECKPOINT_PATH)

In [ ]:
artifact = wandb.Artifact(
    name=f"{RUN_NAME}_best_checkpoint",
    type="model",
    description=(
        "Best ResNet50 attention captioning checkpoint "
        "selected by validation loss."
    ),
)

artifact.add_file(str(RUN_DIR / "checkpoint_best.pth"))
artifact.add_file(str(RUN_DIR / "training_history.csv"))
artifact.add_file(str(RUN_DIR / "vocab.pkl"))

if (RUN_DIR / "bleu_results.csv").exists():
    artifact.add_file(str(RUN_DIR / "bleu_results.csv"))

if (
    RUN_DIR / "sample_predictions_greedy_vs_beam.txt"
).exists():
    artifact.add_file(
        str(
            RUN_DIR / "sample_predictions_greedy_vs_beam.txt"
        )
    )

wandb.log_artifact(artifact)
print("Logged model artifact to W&B.")

In [ ]:
wandb.finish()